# 산불 피해 면적 예측 — EDA와 모델 학습 (Forest Fires)

- 데이터: 포르투갈 공원 산불 517건의 기상·피해 (517행 · 13열)
- 목표: 소실 면적(area, 헥타르) 예측 — 회귀
- 흐름: 불러오기 → 학습 전 확인 → EDA → 학습 → 해석
- 참고: 데이터 소개 data_ForestFires.txt

- 이 데이터의 특징: **예측이 잘 안 되는 게 정상**
  → "데이터에 신호가 없으면 어떤 모델도 못 맞힌다"를 체감

## 1. 불러오기

- 일반 CSV(쉼표) · 헤더 있음 · 결측 없음

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("forestfires.csv")
print(df.shape)          # (517, 13)
df.head()

## 2. 학습 전 확인

- 결측·인코딩은 AutoGluon이 자동
- 이 데이터의 핵심은 **target(area)의 성질**

### 2-1. target의 절반이 0

- area의 47.8%가 0 (517건 중 247건)
- 이 0은 결측이 아니라 '0.1ha 미만의 작은 불'
- 모델이 "대체로 0"만 학습하면 큰 불을 못 맞힘

In [ ]:
print("area 0인 행:", (df["area"]==0).sum(),
      f"({(df['area']==0).mean():.1%})")
print("area 범위:", df["area"].min(), "~", df["area"].max())
# 중앙값 대비 최댓값이 매우 큼 → 극단적으로 치우침

### 2-2. target 분포가 극단적으로 치우침

- 대부분 작고 소수가 매우 큼 (최대 1090ha)
- 로그 변환으로 분포를 완만하게 (0이 많으니 log1p)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,3))
df["area"].hist(bins=50, ax=ax[0]); ax[0].set_title("area (raw)")
np.log1p(df["area"]).hist(bins=50, ax=ax[1])
ax[1].set_title("log1p(area)")
plt.show()

# 로그 변환 컬럼 추가
df["log_area"] = np.log1p(df["area"])

### 2-3. X, Y는 숫자로 코딩된 위치

- 공원 격자 좌표 (X 1~9, Y 2~9)
- 숫자지만 위치 코드 → 범주형이 나을 수도 (비교 대상)

In [ ]:
# 일단 확인만 — 학습 시 범주형/수치형 둘 다 시도해 볼 것
print("X:", sorted(df["X"].unique()))
print("Y:", sorted(df["Y"].unique()))

## 3. EDA

In [ ]:
from data_profiling import ProfileReport

profile = ProfileReport(df, progress_bar=False)
profile.to_file("forestfires_eda.html")   # 브라우저에서 열기

### 3-1. rain은 거의 전부 0

- 강우 컬럼이 98% 이상 0 → 정보가 거의 없음
- 제거를 검토할 수 있다

In [ ]:
print("rain 0비율:", (df["rain"]==0).mean().round(3))
# 거의 전부 0 → 변수로서 가치가 낮음

### 3-2. 기상 변수와 피해 면적

- 기온·습도·바람이 큰 불과 관련이 있는가
- (미리 말하면, 관계가 뚜렷하지 않을 것)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13,3))
for i, c in enumerate(["temp","RH","wind"]):
    df.plot.scatter(x=c, y="log_area", alpha=0.3, ax=ax[i])
plt.show()
# 점들이 흩어져 뚜렷한 패턴이 안 보인다면
# → 이 변수들로 area를 예측하기 어렵다는 신호

## 4. 학습 — 두 가지 접근

- 이 데이터는 회귀가 어려우므로 두 방식을 비교
  (A) 로그 면적 회귀   (B) 큰 불 여부 이진 분류

### 4-1. (A) 로그 면적 회귀

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    df, test_size=0.2, random_state=42)

# area 원본은 제외(log_area가 타깃), big_fire도 아직 없음
cols_A = [c for c in df.columns if c != "area"]
pred_A = TabularPredictor(
    label="log_area",
    eval_metric="root_mean_squared_error",
).fit(train_data[cols_A], presets="medium_quality", time_limit=300)

In [ ]:
print(pred_A.evaluate(test_data[cols_A]))
pred_A.leaderboard(test_data[cols_A])
# 성능이 낮게 나올 것 — 데이터 탓이지 모델 탓이 아니다

### 4-2. (B) 큰 불 여부 이진 분류

- area>0 이면 1 (큰 불), 아니면 0
- 회귀보다 다루기 쉬운 문제로 바꿔봄

In [ ]:
df["big_fire"] = (df["area"] > 0).astype(int)
train_b, test_b = train_test_split(df, test_size=0.2, random_state=42)

cols_B = [c for c in df.columns
          if c not in ["area","log_area"]]
pred_B = TabularPredictor(
    label="big_fire",
    eval_metric="roc_auc",
).fit(train_b[cols_B], presets="medium_quality", time_limit=300)

In [ ]:
print(pred_B.evaluate(test_b[cols_B]))
pred_B.leaderboard(test_b[cols_B])

## 5. 해석

In [ ]:
pred_B.feature_importance(test_b[cols_B])
# FWI 지수(FFMC/DMC/DC/ISI)와 기상값 중 무엇이 중요한가

## 정리

- target의 절반이 0 · 극단적 치우침 → 로그 변환, 분류로 전환
- rain 거의 전부 0 → 정보 적음
- 성능이 낮게 나오는 게 정상 — **데이터에 신호가 부족**
  → 모델을 바꿔도 크게 나아지지 않는다
- "AutoML이 만능은 아니다 · 데이터가 먼저다"를 체감하는 사례

- 다른 데이터와 비교:
  Absenteeism·Crop은 전처리가 핵심,
  Concrete는 파생변수, **Forest Fires는 데이터의 한계**